In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Simulation timing
total_duration = 3.0  # seconds
dt = 0.0001  # seconds (0.1 ms)
time = np.arange(0.0, total_duration, dt)

# LIF parameters
v_rest = -65.0  # mV
v_reset = -70.0  # mV
v_threshold = -50.0  # mV
tau_m = 0.020  # s
r_m = 10.0  # MOhm (effective resistance scaling)

# Input current: 1-second step from 1s to 2s
i_stim = np.zeros_like(time)
stim_amplitude = 2  # nA-equivalent scale
stim_mask = (time >= 1.0) & (time < 2.0)
i_stim[stim_mask] = stim_amplitude

# State variables
v = np.full_like(time, v_rest)
dv_dt_trace = np.zeros_like(time)
spike_times = []

# Euler integration of LIF differential equation
for i in range(1, len(time)):
    dv_dt = (-(v[i - 1] - v_rest) + r_m * i_stim[i - 1]) / tau_m
    dv_dt_trace[i] = dv_dt
    v[i] = v[i - 1] + dt * dv_dt

    if v[i] >= v_threshold:
        spike_times.append(time[i])
        v[i] = v_reset

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

ax1.plot(time, v, color="tab:blue", linewidth=1.0, label="Membrane potential")
ax1.axhline(
    v_threshold, color="tab:red", linestyle="--", linewidth=1.0, label="Threshold"
)
ax1.set_ylabel("V (mV)")
ax1.set_title("Leaky Integrate-and-Fire Neuron")
ax1.legend(loc="upper right")

ax2.plot(time, i_stim, color="tab:green", linewidth=1.0, label="Input current")
ax2.set_ylabel("I (a.u.)")
ax2.legend(loc="upper right")

ax3.plot(time, dv_dt_trace, color="tab:orange", linewidth=1.0, label="dV/dt")
ax3.set_xlabel("Time (s)")
ax3.set_ylabel("dV/dt (mV/s)")
ax3.legend(loc="upper right")

plt.tight_layout()
plt.show()

print(f"Total spikes: {len(spike_times)}")
if spike_times:
    print("First few spike times (s):", np.array(spike_times[:10]))

In [ ]:
def fit_a_b_c(dv_dt_trace, v, i_stim):
    """Fit constants a, b, c in dv_dt = (-(v - v_rest) + r_m * i_stim) / tau_m."""
    dv_dt_trace = np.asarray(dv_dt_trace)
    v = np.asarray(v)
    i_stim = np.asarray(i_stim)

    if not (dv_dt_trace.shape == v.shape == i_stim.shape):
        raise ValueError("dv_dt_trace, v, and i_stim must have the same shape.")

    # Linearized model: dv_dt = p * v + q * i_stim + r
    # Then p = -1/c, q = a/c, r = b/c.
    y = dv_dt_trace
    x = np.column_stack((v, i_stim, np.ones_like(i_stim)))

    params, _, _, _ = np.linalg.lstsq(x, y, rcond=None)
    p_hat, q_hat, r_hat = params

    if np.isclose(p_hat, 0.0):
        raise ValueError(
            "Could not estimate c because fitted coefficient of v is near zero."
        )

    c_hat = -1.0 / p_hat
    a_hat = q_hat * c_hat
    b_hat = r_hat * c_hat
    return a_hat, b_hat, c_hat

In [ ]:
a_hat, b_hat, c_hat = fit_a_b_c(dv_dt_trace[:], v[:], i_stim[:])
print(f"Estimated r_m: {a_hat:.6f} with bias {100 * (a_hat - r_m) / r_m:.6f} %")
print(
    f"Estimated v_rest: {b_hat:.6f} with bias {100 * (b_hat - v_rest) / v_rest:.6f} %"
)
print(f"Estimated tau_m: {c_hat:.6f} with bias {100 * (c_hat - tau_m) / tau_m:.6f} %")